In [2]:
import pandas as pd

# Load the CSV file
# Assuming the first row contains the actual questions as headers
df = pd.read_csv("./ResultData/SurvayResult.csv")

# --- 定数定義 ---
CATEGORY_COLUMNS = {
    'sus':     list(range(1, 11)),   # SUS 1–10
    'tam':     list(range(11, 14)),  # TAM 11–13
    'trust':   list(range(14, 18)),  # Trust 14–17
    'trust2':  list(range(18, 29)),  # Trust2 18–28
    'nasa':    list(range(29, 35)),  # NASA-TLX 29–37
}

print("--- 質問内容 ---")
for category, q_numbers in CATEGORY_COLUMNS.items():
    print(f"\n## カテゴリ: {category.upper()}")
    print("---")
    for q_num in q_numbers:
        # The column index for the questions starts from 1 (after 'タイムスタンプ').
        # So, the column with question 1 is at index 1, question 2 at index 2, and so on.
        # This assumes your CSV structure where questions Q1-Q37 correspond to actual column positions.
        # If 'タイムスタンプ' is the first column (index 0), then Q1 is at index 1.
        # We also need to handle the case where the header might have leading/trailing spaces or special characters.
        
        # Adjusting for 0-based indexing and the 'タイムスタンプ' column
        # q_num 1 corresponds to df.columns[1], q_num 2 to df.columns[2], etc.
        column_index = q_num

        if column_index < len(df.columns):
            question_text = df.columns[column_index]
            print(f"Q{q_num}: {question_text.strip()}") # .strip() removes leading/trailing whitespace
        else:
            print(f"警告: 質問番号 {q_num} に対応するカラムがデータフレームに見つかりません。")

--- 質問内容 ---

## カテゴリ: SUS
---
Q1: I think that I would like to use this system frequently.
（このシステムを頻繁に使用したいと思います。）
Q2: I found the system unnecessarily complex.
（このシステムは不必要に複雑だと感じました。）
Q3: I thought the system was easy to use.
（このシステムは簡単に使えると感じました。）
Q4: I think that I would need the support of a technical person to be able to use this system.
（このシステムを使うには技術的なサポートが必要だと思いました。）
Q5: I found the various functions in this system were well integrated.
（このシステムのさまざまな機能はうまく統合されていると感じました。）
Q6: I thought there was too much inconsistency in this system.
（このシステムには一貫性がないと感じました。）
Q7: I would imagine that most people would learn to use this system very quickly.
（ほとんどの人がこのシステムをすぐに使いこなせると思います。）
Q8: I found the system very cumbersome to use.
（このシステムは使いづらいと感じました。）
Q9: I felt very confident using the system.
（このシステムを使用する際、自信を持って使えました。）
Q10: I needed to learn a lot of things before I could get going with this system.
（このシステムを使い始める前に、多くのことを学ぶ必要がありました。）

## カテゴリ: TAM
---
Q11: Using this interaction method imp

In [6]:
import pandas as pd

# 元のCSVを読み込み
df = pd.read_csv("SurvayResult.csv")

# index 137以降を抽出
subset_df = df.iloc[136:]   # 137行目以降すべて

# 新しいCSVとして保存
subset_df.to_csv("SurvayResult.csv", index=False)

print("✅ 新しいCSVファイル 'SurvayResult.csv' を作成しました。")


✅ 新しいCSVファイル 'SurvayResult.csv' を作成しました。


In [9]:
import pandas as pd

# --- CSV読み込み ---
survey = pd.read_csv("SurvayResult.csv")
latin = pd.read_csv("../latin_square.csv")

# もし文字化けがある場合 (BOM付きなど)
# latin = pd.read_csv("latin_square.csv", encoding="utf-8-sig")

# --- Condition列を数値化（全角や文字を除去） ---
survey["Condition"] = (
    survey["Condition"]
    .astype(str)
    .str.replace("４", "4")
    .str.replace("３", "3")
    .str.replace("２", "2")
    .str.replace("１", "1")
    .str.extract("(\d)")
    .astype(float)
)

# --- 各被験者ごと（4行ずつ）に検証 ---
n_participants = len(survey) // 4
results = []

for i in range(n_participants):
    # 被験者ごとのsurveyデータ
    sub = survey.iloc[i*4:(i+1)*4].reset_index(drop=True)
    
    # latin_square の ConditionNum列を抽出
    latin_row = latin.iloc[i]
    latin_order = [
        latin_row["Condition 1_ConditionNum"],
        latin_row["Condition 2_ConditionNum"],
        latin_row["Condition 3_ConditionNum"],
        latin_row["Condition 4_ConditionNum"]
    ]
    
    # 一致チェック
    is_match = list(sub["Condition"]) == latin_order
    results.append({
        "Participant": latin_row["Participant"],
        "SurveyCondition": list(sub["Condition"]),
        "LatinOrder": latin_order,
        "Match": is_match
    })

# --- 結果をDataFrame化 ---
check_df = pd.DataFrame(results)

# 不一致の人を表示
print("⚠️ 一致していない被験者一覧:")
print(check_df[check_df["Match"] == False])

# 全体の一致率
print(f"\n✅ 一致率: {(check_df['Match'].sum() / len(check_df)) * 100:.1f}%")


⚠️ 一致していない被験者一覧:
   Participant       SurveyCondition    LatinOrder  Match
9          P10  [1.0, 4.0, 3.0, 2.0]  [2, 3, 4, 1]  False
21         P22  [3.0, 2.0, 4.0, 1.0]  [2, 3, 4, 1]  False

✅ 一致率: 91.7%


In [33]:
# build_summary_from_raw.py
# -*- coding: utf-8 -*-
"""
SurvayResult.csv（1行=参加者×条件の生回答）から、
SUS / TAM(PEOU, PU, Total) / TPA / NASA(6次元+Overall) を番号で集計し、
outputs/summary_all.csv に書き出す。

実行:
  python build_summary_from_raw.py
"""

import os
import numpy as np
import pandas as pd

# ===== あなたの SurvayResult.csv の「列番号(0始まり)」に必ず合わせる！ =====
SCHEMA = {
    "participant_col": 1,        # 例: ID / participant の列
    "condition_col":   2,        # 例: Condition の列（1..4 or 日本語/英語）

    # 設問の列レンジ（0始まり, rangeは終端含まないPython標準ではなく「明示リスト」にしている）
    "SUS_cols":  list(range(3, 13)),      # SUS 10項目 (1-5)
    "TAM_cols":  list(range(13, 25)),     # TAM 12項目 (1-7) → PEOU=前6, PU=後6
    "TPA_cols":  list(range(25, 37)),     # TPA 12項目 (1-7) → 前7=ポジ / 後5=ネガ(反転)
    "NASA_cols": list(range(37, 43)),     # NASA-TLX 6項目 (1-21) [Mental..Frustration]
}

# Condition 正規化（数値/日本語/英語 → SR/P+SR/Pointing/Label）
COND_ABBR = {1: "SR", 2: "P+SR", 3: "Pointing", 4: "Label"}
COND_ALIAS = {
    "SR": "SR", "SpatialReference": "SR", "空間参照だけ": "SR",
    "P+SR": "P+SR", "SpatialReference+Pointing": "P+SR", "Pointing + 空間参照": "P+SR",
    "Pointing": "Pointing",
    "Label": "Label", "ラベル": "Label",
}

BASE   = os.path.abspath(os.getcwd())
CSV_IN = os.path.join(BASE, "ResultData", "SurvayResult.csv")
OUTDIR = os.path.join(BASE, "outputs")
CSV_OUT = os.path.join(OUTDIR, "summary_all.csv")
os.makedirs(OUTDIR, exist_ok=True)

# ===== スコアリング =====
def sus_score(items) -> float:
    """SUS: 奇数(r-1), 偶数(5-r), 合計×2.5 → 0-100"""
    xs = [float(x) for x in items]
    s = [(r - 1) if (i % 2 == 1) else (5 - r) for i, r in enumerate(xs, start=1)]
    return float(np.sum(s) * 2.5)

def tam_score(items) -> dict:
    """TAM: 12項目 (1-7) → PEOU=前6平均, PU=後6平均, Total=全体平均"""
    xs = np.asarray([float(x) for x in items], float)
    return {
        "PEOU":  float(np.mean(xs[:6])),
        "PU":    float(np.mean(xs[6:12])),
        "Total": float(np.mean(xs[:12])),
    }

def tpa_score(items) -> float:
    """TPA: 12項目 (1-7) → 前7=ポジティブ, 後5=ネガティブを反転(8-r)して全体平均"""
    xs = np.asarray([float(x) for x in items], float)
    pos = xs[:7]
    neg = 8.0 - xs[7:12]    # 後半5を反転
    allv = np.concatenate([pos, neg])
    return float(np.mean(allv))

def nasatlx_score_raw(items) -> dict:
    """
    あなたの既存表に合わせる：
    - Performance は反転しない（そのままの値）
    - Overall は6次元の生値平均
    """
    xs = np.asarray([float(x) for x in items], float)
    dims = {
        "Mental":       float(xs[0]),
        "Physical":     float(xs[1]),
        "Temporal":     float(xs[2]),
        "Performance":  float(xs[3]),  # 反転しない
        "Effort":       float(xs[4]),
        "Frustration":  float(xs[5]),
    }
    dims["Overall"] = float(np.mean(list(dims.values())))
    return dims

def normalize_condition(v):
    try:
        i = int(v)
        if i in COND_ABBR:
            return COND_ABBR[i]
    except Exception:
        pass
    key = str(v).strip()
    return COND_ALIAS.get(key, key)

def main():
    df = pd.read_csv(CSV_IN, header=0)

    pc, cc = SCHEMA["participant_col"], SCHEMA["condition_col"]
    sus_cols, tam_cols, tpa_cols, nasa_cols = (
        SCHEMA["SUS_cols"], SCHEMA["TAM_cols"], SCHEMA["TPA_cols"], SCHEMA["NASA_cols"]
    )

    rows = []
    for i in range(len(df)):
        r = df.iloc[i]

        participant = str(r.iloc[pc]).strip()      # e.g., "P1"
        condition   = normalize_condition(r.iloc[cc])

        sus_items  = [r.iloc[c] for c in sus_cols]
        tam_items  = [r.iloc[c] for c in tam_cols]
        tpa_items  = [r.iloc[c] for c in tpa_cols]
        nasa_items = [r.iloc[c] for c in nasa_cols]

        sus = sus_score(sus_items)
        tam = tam_score(tam_items)
        tpa = tpa_score(tpa_items)
        tlx = nasatlx_score_raw(nasa_items)

        rows.append({
            "participant": participant,
            "condition":   condition,

            "SUS":         sus,

            # TAM
            "TAM_PEOU":    tam["PEOU"],
            "TAM_PU":      tam["PU"],
            "TAM":   tam["Total"],

            # TPA
            "TPA":   tpa,

            # NASA
            "NASA_Mental":       tlx["Mental"],
            "NASA_Physical":     tlx["Physical"],
            "NASA_Temporal":     tlx["Temporal"],
            "NASA_Performance":  tlx["Performance"],  # 反転なし
            "NASA_Effort":       tlx["Effort"],
            "NASA_Frustration":  tlx["Frustration"],
            "NASA_Overall":      tlx["Overall"],
        })

    out = pd.DataFrame.from_records(rows)

    # 並びと通し番号（あなたの表に合わせて）
    cond_cat = pd.CategoricalDtype(categories=["SR","P+SR","Pointing","Label"], ordered=True)
    out["condition"] = out["condition"].astype(cond_cat)
    out = out.sort_values(["participant", "condition"]).reset_index(drop=True)
    out.insert(0, "row_id", np.arange(1, len(out) + 1).astype(int))

    # 列順（あなたの元表 + TAM/TPAを末尾に追加）
    cols = [
        "row_id", "participant", "condition", "SUS",
        "NASA_Mental","NASA_Physical","NASA_Temporal","NASA_Performance",
        "NASA_Effort","NASA_Frustration","NASA_Overall",
        "TAM_PEOU","TAM_PU","TAM","TPA",
    ]
    out = out[cols]

    out.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
    print(f"✅ Wrote: {CSV_OUT}  (rows={len(out)})")

main()


✅ Wrote: /home/tenma/school/InteractiveSmartHome/Experiment/Results/outputs/summary_all.csv  (rows=96)


In [185]:
import pandas as pd
import os

# 入力ファイルパス
file_path = './ResultData/SurvayResult_Updated.csv'


df = pd.read_csv(file_path)
extraction = df[["ID", "Condition","Free Description (自由記述)"]]
condition_not_nan = extraction['Free Description (自由記述)'].notna()

# 条件2: 'Free Description (自由記述)' が 'Nah' という文字列ではない
condition_not_nah = extraction['Free Description (自由記述)'] != 'Nah'

# 両方の条件を満たす行だけを抽出
final_df = extraction[condition_not_nan & condition_not_nah]

# 結果の確認
final_df[final_df["ID"]=="P3"].to_string(index=False)

'ID  Condition                                                                                Free Description (自由記述)\nP3          4                                                        覚えればこれが一番いいと思った。\\n覚えることが負担で、人によって違うんじゃないかなと思った。\nP3          1 言葉でいうと便利な反面、\\n言葉で説明しずらいこともあった。\\n赤色の枠が見えにくい\\n\\nI...\\n                音声のみでは明らかにできそうなタスクとできなそうなタスクの差があった'

In [191]:
# 置き換えのルールを辞書で定義
condition_map = {
    1: 'SR',
    2: 'P+SR',
    3: 'Pointing',
    4: 'Label'
}
pd.set_option('display.max_colwidth', None)
# 'Condition' 列に map を適用して置き換え
# .copy() をつけて、後続の処理での警告 (SettingWithCopyWarning) を回避
relabeled_df = final_df.copy() # または final_df = extraction.copy()
relabeled_df['Condition'] = final_df['Condition'].map(condition_map)

relabeled_df[relabeled_df["Condition"]=="SR"]

,ID,Condition,Free Description (自由記述)
10,P3,SR,言葉でいうと便利な反面、\n言葉で説明しずらいこともあった。\n赤色の枠が見えにくい\n\nI...\n 音声のみでは明らかにできそうなタスクとできなそうなタスクの差があった
23,P6,SR,空間認識という点から、自身からの距離や方向が主要な情報として扱われる要素であるように感じるが、対象のライトの数が多く、同じような距離、方向に複数のライトがあった場合にそれらを区別するように指示を出すのが難しいように感じた。次に選択肢を視界に入るものに絞ることでライトの指定ができるが、視界の境界があいまいで、かつ処理の間一点を見つめ続けなければいけないことから身体的な負担も増加するように感じた。自然と指示が長くなることが多くなり、結果としてLLMの処理にかかる時間もほかのタスク時よりも比較的多くなるようにも感じた。家庭内などライトがそこまで多くない環境ではある程度性能が発揮されるかもしれないが、ライトの数や配置で難易度が増大し汎用性が下がるように感じた。
26,P7,SR,自分の視界を基準とすることで正確に操作をすることができるが、身体的な負担が大きくなるという懸念がある
36,P10,SR,「正面」の認識精度が悪い。ずれていると感じた。
42,P11,SR,照明の位置を覚えて命令するよりも直感的なので慣れれば、2つ目の位置を覚えるやり方よりも使いやすくなると思います
45,P12,SR,ライトが複数ある環境で、この操作使いずらいと思いました。一般の家庭なら便利に使えると思いました
48,P13,SR,真ん中といったときに自分が思った配列の真ん中でなく部屋の真ん中として認識されたタスクがあったのが気になった
74,P19,SR,「視界の中央」が一番通りやすかったので、数が少ないものは全部それでやりました
77,P20,SR,使いやすかったが、本当にこの指示で大丈夫なのか自信を持てないときもあった。\nまた、意図しない電気がついてしまうこともあった\nしかし、まとめてつけれられるのはとても便利に感じた
80,P21,SR,部屋の状態を理解しているのか不安になった。


In [168]:
import pandas as pd
import os

# 入力ファイルパス
file_path = './ResultData/Old_SurvayResult.csv'


df = pd.read_csv(file_path)
extraction = df[["ID", "Condition","Free Description (自由記述)"]]
condition_not_nan = extraction['Free Description (自由記述)'].notna()

# 条件2: 'Free Description (自由記述)' が 'Nah' という文字列ではない
condition_not_nah = extraction['Free Description (自由記述)'] != 'Nah'

# 両方の条件を満たす行だけを抽出
final_df = extraction[condition_not_nan & condition_not_nah]

# 結果の確認
cond= final_df[(final_df["ID"]=="P2") & (final_df["Condition"] == 2)][free_col].to_string(index=False)
# final_df[(final_df["ID"]=="P2")]
cond

'覚えればこれが一番いいと思った。\\n覚えることが負担で、人によって違うんじゃないかなと思った。'

In [181]:
# 置き換えのルールを辞書で定義
condition_map = {
    1: 'SR',
    2: 'P+SR',
    3: 'Pointing',
    4: 'Label'
}
free_col = "Free Description (自由記述)"
# 'Condition' 列に map を適用して置き換え
# .copy() をつけて、後続の処理での警告 (SettingWithCopyWarning) を回避
relabeled_df = final_df.copy() # または final_df = extraction.copy()
relabeled_df['Condition'] = final_df['Condition'].map(condition_map)

relabeled_df[ (relabeled_df["ID"] == "P2") ]

import os

# 入力ファイルパス
file_path = './ResultData/Old_SurvayResult.csv'


df = pd.read_csv(file_path)
extraction = df[["ID", "Condition","Free Description (自由記述)"]]
condition_not_nan = extraction['Free Description (自由記述)'].notna()

# 条件2: 'Free Description (自由記述)' が 'Nah' という文字列ではない
condition_not_nah = extraction['Free Description (自由記述)'] != 'Nah'

# 両方の条件を満たす行だけを抽出
final_df = extraction[condition_not_nan & condition_not_nah]
new = pd.read_csv('./ResultData/SurvayResult.csv')
# 結果の確認
cond= final_df[(final_df["ID"]=="P2") & (final_df["Condition"] ==2)][free_col].to_string(index=False)
# final_df[(final_df["ID"]=="P2")]
cond


new.loc[(new["ID"] == "P3") & (new["Condition"] == 4), free_col] = cond


# 結果の確認
cond= final_df[(final_df["ID"]=="P2") & (final_df["Condition"] ==1)][free_col].to_string(index=False)
# final_df[(final_df["ID"]=="P2")]
cond


new.loc[(new["ID"] == "P3") & (new["Condition"] == 1), free_col] = cond

# new[(new["ID"] == "P3") & (new["Condition"] == 1)][["ID", "Condition", free_col]]

new.to_csv("./ResultData/SurvayResult_Updated.csv")

In [172]:
import pandas as pd import os # --- 1. ファイルパスの定義 --- old_file_path = './ResultData/Old_SurvayResult.csv' new_file_path = './ResultData/SurvayResult.csv' # 保存するファイルパス（元ファイルを上書きしないよう、別名で保存します） output_file_path = './ResultData/SurvayResult_Updated.csv' # --- 2. 自由記述コラム名の定義 --- # ※※※ ここの名前がCSVファイルと一致しているか確認してください ※※※ free_text_column = 'Free Description (自由記述)' # --- 3. 両方のCSVファイルを読み込む --- df_old = pd.read_csv(old_file_path) df_new = pd.read_csv(new_file_path) for i in range(1,5): cond = df_old[df_old["ID"]=="P2"][["ID", "Condition", free_text_column]].iloc[:4]ne

SyntaxError: invalid syntax (1321531334.py, line 1)

,ID,Condition,Free Description (自由記述)
8,P3,3,NaN
9,P3,4,NaN
10,P3,1,言葉でいうと便利な反面、\n言葉で説明しずらいこともあった。\n赤色の枠が見えにくい\n\n...
11,P3,2,NaN
